# CNN Aircraft Image Classification

This project implements convolutional neural networks (CNNs) to classify
aircraft images from the FGVC-Aircraft dataset.

The project compares different CNN architectures and evaluates their
performance using accuracy metrics and visualization techniques.

## Dataset overview: FGVC-Aircraft dataset

Public FGVC-Aircraft 2013b dataset
Real photos of aircraft taken “in the wild”
High resolution (≈1–2 MP), varied: different airports, lighting, weather, viewpoints
Each image shows one main aircraft + background (sky / runway / clutter) --> realistic conditions.

## Implemented Models
3 CNN setups: 
- Shallow CNN
- Deep CNN
- Deep CNN with Regularization

## Evaluation Metrics

- Top-1 Accuracy
- Top-3 Accuracy
- Confusion Matrix
- Grad-CAM


## Selected classes , sample sizes and preprocessing
Subset of 10 aircraft families
For each family, a minimum number of images per split is required: (train ≥ 60, val ≥ 15, test ≥ 15) -->  avoids tiny classes that are impossible
to learn

Automatically selected 10 families that satisfy this:
 A320, A340
 Boeing 737, 747, 757, 767
 Embraer E-Jet, Dash 8, Gulfstream, MD-80
Then balanced sampling:
 Train: ~67–120 images per class (total 948 images)
 Val: 40 images per class (total 400)
 Test: 40 images per class (total 400)

Image preprocessing
Each original image has a 20-pixel copyright banner at the bottom
→ we crop it off (to avoid learning text or borders)
Images are resized to 160 × 160 pixels


## Example Results

[Confusion Matrix]

[GradCAM]

In [ ]:

"""
FGVC-Aircraft CNN Classification (cropped + tuned + report-ready)

Main features:
- Uses official FGVC-Aircraft splits at "family" label level
- Auto-selects 10 well-populated families
- Crops images to bounding boxes (aircraft only) using images_box.txt
- Defines 3 CNN setups:
    * Shallow
    * Deep
    * Deep + Regularization (dropout + weight decay)
- Trains all 3 (with tuned hyperparameters)
- Saves accuracy & loss curves (train/val)
- Computes Top-1 and Top-3 accuracy on test
- Confusion matrices for ALL three setups
- Example predictions (correct high/low confidence + wrong)
- Grad-CAM visualizations on best model
"""

import os
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
from torch.utils.data import DataLoader
from PIL import Image

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
)

# For showing some sample crops in Jupyter/Spyder
try:
    from IPython.display import display
except ImportError:
    display = print  


# ============================================================
# Config
# ============================================================
SEED = 42

BASE_DIR = "fgvc-aircraft-2013b/data"
IMAGE_DIR = os.path.join(BASE_DIR, "images")
OUT_DIR = "data"
FIG_DIR = "report_figures"
BOX_FILE = os.path.join(BASE_DIR, "images_box.txt")

LABEL_LEVEL = "family"  # "variant" / "family" / "manufacturer"

IMG_SIZE = 160
BATCH_SIZE = 16

# --- Tuned training settings ---
EPOCHS = 30          # was 20
PATIENCE = 7         # was 5

LR_SHALLOW = 1e-3
LR_DEEP = 5e-4       # smaller for deeper models
LR_DEEP_REG = 5e-4

WEIGHT_DECAY_DEEP_REG = 5e-5  

BANNER_PX = 20  # not used anymore after cropping, kept for documentation

K_CLASSES = 10
TRAIN_SAMPLES = 120
VAL_SAMPLES   = 40
TEST_SAMPLES  = 40

MIN_TRAIN = 80
MIN_VAL   = 20
MIN_TEST  = 20


# ============================================================
# Reproducibility
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

os.makedirs(FIG_DIR, exist_ok=True)
assert os.path.exists(IMAGE_DIR), f"Images folder not found: {IMAGE_DIR}"


# ============================================================
# Files for chosen label level
# ============================================================
LEVEL_TO_FILES = {
    "variant": (
        "images_variant_train.txt",
        "images_variant_val.txt",
        "images_variant_test.txt",
    ),
    "family": (
        "images_family_train.txt",
        "images_family_val.txt",
        "images_family_test.txt",
    ),
    "manufacturer": (
        "images_manufacturer_train.txt",
        "images_manufacturer_val.txt",
        "images_manufacturer_test.txt",
    ),
}

TRAIN_FILE, VAL_FILE, TEST_FILE = LEVEL_TO_FILES[LABEL_LEVEL]


# ============================================================
# Utilities: labels, boxes, sampling, export
# ============================================================
def load_label_file(path: str) -> pd.DataFrame:
    """Parse files like images_family_train.txt: first token is image_id,
    rest is the (possibly multi-word) label."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            image_id = parts[0]
            label = " ".join(parts[1:])
            data.append((image_id, label))
    return pd.DataFrame(data, columns=["image_id", "label"])


def clean_out_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def sample_per_class(df, labels, n):
    """Stratified sampling: up to n samples per class."""
    df = df[df["label"].isin(labels)].copy()
    sampled = (
        df.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n), random_state=SEED))
        .reset_index(drop=True)
    )
    return sampled


def load_boxes(path):
    """
    Load bounding boxes from images_box.txt.
    Returns dict: image_id -> (left, upper, right, lower) in 0-based coords.

    Original file uses 1-based coordinates (xmin, ymin, xmax, ymax).
    PIL expects (left, upper, right, lower), right/lower are exclusive,
    so we convert (xmin, ymin, xmax, ymax) -> (xmin-1, ymin-1, xmax, ymax).
    """
    boxes = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            img_id = parts[0]
            xmin, ymin, xmax, ymax = map(int, parts[1:])
            left = xmin - 1
            upper = ymin - 1
            right = xmax
            lower = ymax
            boxes[img_id] = (left, upper, right, lower)
    return boxes


BOXES = load_boxes(BOX_FILE)
print(f"Loaded {len(BOXES)} bounding boxes.")


def export_split(df, split):
    """
    Export images into OUT_DIR/split/label/*.jpg

    IMPORTANT: here we apply the bounding-box crop so models see mainly aircraft.
    """
    missing = 0
    cropped = 0

    for _, row in df.iterrows():
        img_id = row.image_id
        src = os.path.join(IMAGE_DIR, f"{img_id}.jpg")
        if not os.path.exists(src) or img_id not in BOXES:
            missing += 1
            continue

        dst_dir = os.path.join(OUT_DIR, split, row.label)
        os.makedirs(dst_dir, exist_ok=True)
        dst = os.path.join(dst_dir, f"{img_id}.jpg")

        with Image.open(src) as im:
            im = im.convert("RGB")
            bbox = BOXES[img_id]
            im = im.crop(bbox)
            im.save(dst)

        cropped += 1

    print(f"[{split}] cropped+saved={cropped}, missing_skipped={missing}")


def count_split(split):
    base = os.path.join(OUT_DIR, split)
    print(f"\n{split.upper()} counts:")
    for cls in sorted(os.listdir(base)):
        n = len(os.listdir(os.path.join(base, cls)))
        print(f"  {cls}: {n}")


def choose_valid_classes(train_df, val_df, test_df, k, min_train, min_val, min_test):
    train_counts = train_df["label"].value_counts()
    val_counts = val_df["label"].value_counts()
    test_counts = test_df["label"].value_counts()

    eligible = []
    for lbl, ntr in train_counts.items():
        if ntr < min_train:
            continue
        if val_counts.get(lbl, 0) < min_val:
            continue
        if test_counts.get(lbl, 0) < min_test:
            continue
        eligible.append(lbl)
    return eligible[:k], (train_counts, val_counts, test_counts)


def inspect_thresholds(train_counts, val_counts, test_counts):
    combos = [(80, 20, 20), (60, 15, 15), (40, 10, 10), (30, 8, 8), (20, 5, 5)]
    print("\nEligibility check (how many classes meet thresholds):")
    for mt, mv, mte in combos:
        eligible = 0
        for lbl, ntr in train_counts.items():
            if (
                ntr >= mt
                and val_counts.get(lbl, 0) >= mv
                and test_counts.get(lbl, 0) >= mte
            ):
                eligible += 1
        print(f"  train>={mt}, val>={mv}, test>={mte} -> eligible classes: {eligible}")


# ============================================================
# Load official splits & pick classes
# ============================================================
train_df = load_label_file(os.path.join(BASE_DIR, TRAIN_FILE))
val_df = load_label_file(os.path.join(BASE_DIR, VAL_FILE))
test_df = load_label_file(os.path.join(BASE_DIR, TEST_FILE))

classes, counts = choose_valid_classes(
    train_df, val_df, test_df, K_CLASSES, MIN_TRAIN, MIN_VAL, MIN_TEST
)
train_counts, val_counts, test_counts = counts

print(f"\nLabel level: {LABEL_LEVEL}")
print(
    f"Unique labels: train={train_df['label'].nunique()}, "
    f"val={val_df['label'].nunique()}, test={test_df['label'].nunique()}"
)
inspect_thresholds(train_counts, val_counts, test_counts)

if len(classes) < K_CLASSES:
    print(
        f"\nOnly found {len(classes)} classes with MIN thresholds. "
        "Relaxing thresholds automatically..."
    )
    for (mt, mv, mte) in [(60, 15, 15), (40, 10, 10), (30, 8, 8), (20, 5, 5)]:
        classes, _ = choose_valid_classes(
            train_df, val_df, test_df, K_CLASSES, mt, mv, mte
        )
        if len(classes) >= 2:
            MIN_TRAIN, MIN_VAL, MIN_TEST = mt, mv, mte
            break

print(
    f"\nUsing thresholds: train>={MIN_TRAIN}, val>={MIN_VAL}, test>={MIN_TEST}"
)
classes, _ = choose_valid_classes(
    train_df, val_df, test_df, K_CLASSES, MIN_TRAIN, MIN_VAL, MIN_TEST
)

print("\nSelected classes:")
for c in classes:
    print(" -", c)


# ============================================================
# Sample and export dataset (cropped to bounding boxes)
# ============================================================
train_df_s = sample_per_class(train_df, classes, TRAIN_SAMPLES)
val_df_s = sample_per_class(val_df, classes, VAL_SAMPLES)
test_df_s = sample_per_class(test_df, classes, TEST_SAMPLES)

clean_out_dir(OUT_DIR)
export_split(train_df_s, "train")
export_split(val_df_s, "val")
export_split(test_df_s, "test")

count_split("train")
count_split("val")
count_split("test")


# ---- Visual sanity check: show a few cropped training + test images ----
def show_some_crops(split="train", n=5):
    base = os.path.join(OUT_DIR, split)
    if not os.path.exists(base):
        print(f"No split '{split}' found at {base}")
        return
    classes_local = sorted(os.listdir(base))
    print(f"\nShowing {n} random cropped images from '{split}' split:")
    for _ in range(n):
        cls = random.choice(classes_local)
        folder = os.path.join(base, cls)
        fname = random.choice(os.listdir(folder))
        path = os.path.join(folder, fname)
        img = Image.open(path).convert("RGB")
        print(f"  class={cls}, file={fname}, size={img.size}")
        try:
            display(img)
        except Exception:
            pass



show_some_crops("train", n=5)
show_some_crops("test", n=5)


# ============================================================
# Transforms (no banner removal needed after crop)
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),                        # small rotation
    transforms.ColorJitter(brightness=0.1, contrast=0.1),         # light color aug
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

train_data = datasets.ImageFolder(
    os.path.join(OUT_DIR, "train"), transform=train_transform
)
val_data = datasets.ImageFolder(
    os.path.join(OUT_DIR, "val"), transform=eval_transform
)
test_data = datasets.ImageFolder(
    os.path.join(OUT_DIR, "test"), transform=eval_transform
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

NUM_CLASSES = len(train_data.classes)
print("\nNUM_CLASSES:", NUM_CLASSES)


# ============================================================
# Models
# ============================================================
class CNN_Shallow(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        s = IMG_SIZE // 4  # after 2x MaxPool(2)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * s * s, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class CNN_Deep(nn.Module):
    def __init__(self, num_classes, dropout=False):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        s = IMG_SIZE // 8  # 3x MaxPool(2)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * s * s, 256), nn.ReLU(),
            nn.Dropout(0.3) if dropout else nn.Identity(),  # tuned dropout
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ============================================================
# Training helpers
# ============================================================
def batch_acc(logits, y):
    return (logits.argmax(1) == y).float().mean().item()


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    losses, accs = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        losses.append(loss.item())
        accs.append(batch_acc(logits, y))
    return float(np.mean(losses)), float(np.mean(accs))


def train_model(model, optimizer, criterion, epochs=EPOCHS, patience=PATIENCE):
    model.to(device)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    best_val_loss = float("inf")
    best_state = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        tr_losses, tr_accs = [], []

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            tr_losses.append(loss.item())
            tr_accs.append(batch_acc(logits, y))

        val_loss, val_acc = evaluate(model, val_loader, criterion)

        hist["train_loss"].append(float(np.mean(tr_losses)))
        hist["train_acc"].append(float(np.mean(tr_accs)))
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)

        print(
            f"Epoch {ep:02d} | train loss {hist['train_loss'][-1]:.4f} "
            f"acc {hist['train_acc'][-1]:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, hist


# ============================================================
# Train all three models (with tuned hyperparameters)
# ============================================================
criterion = nn.CrossEntropyLoss()

# Shallow
model_A = CNN_Shallow(NUM_CLASSES)
optimizer_A = optim.Adam(model_A.parameters(), lr=LR_SHALLOW)
model_A, hist_A = train_model(model_A, optimizer_A, criterion)

# Deep (no reg)
model_B = CNN_Deep(NUM_CLASSES, dropout=False)
optimizer_B = optim.Adam(model_B.parameters(), lr=LR_DEEP)
model_B, hist_B = train_model(model_B, optimizer_B, criterion)

# Deep + Reg
model_C = CNN_Deep(NUM_CLASSES, dropout=True)
optimizer_C = optim.Adam(
    model_C.parameters(),
    lr=LR_DEEP_REG,
    weight_decay=WEIGHT_DECAY_DEEP_REG,
)
model_C, hist_C = train_model(model_C, optimizer_C, criterion)


# ============================================================
# Save accuracy / loss curves
# ============================================================
def save_curves(hist_A, hist_B, hist_C):
    colors = {
        "Shallow": "tab:blue",
        "Deep": "tab:orange",
        "Deep+Reg": "tab:green",
    }

    # ---- Accuracy curves ----
    plt.figure(figsize=(8, 5))

    plt.plot(hist_A["train_acc"], color=colors["Shallow"], linestyle="-", label="Shallow train")
    plt.plot(hist_A["val_acc"],   color=colors["Shallow"], linestyle="--", marker="o",
             markersize=4, label="Shallow val")

    plt.plot(hist_B["train_acc"], color=colors["Deep"], linestyle="-", label="Deep train")
    plt.plot(hist_B["val_acc"],   color=colors["Deep"], linestyle="--", marker="s",
             markersize=4, label="Deep val")

    plt.plot(hist_C["train_acc"], color=colors["Deep+Reg"], linestyle="-", label="Deep+Reg train")
    plt.plot(hist_C["val_acc"],   color=colors["Deep+Reg"], linestyle="--", marker="^",
             markersize=4, label="Deep+Reg val")

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy vs Epochs (Train & Validation)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "fig_accuracy_curves.png"), dpi=200)
    plt.show()

    # ---- Loss curves ----
    plt.figure(figsize=(8, 5))

    plt.plot(hist_A["train_loss"], color=colors["Shallow"], linestyle="-", label="Shallow train")
    plt.plot(hist_A["val_loss"],   color=colors["Shallow"], linestyle="--", marker="o",
             markersize=4, label="Shallow val")

    plt.plot(hist_B["train_loss"], color=colors["Deep"], linestyle="-", label="Deep train")
    plt.plot(hist_B["val_loss"],   color=colors["Deep"], linestyle="--", marker="s",
             markersize=4, label="Deep val")

    plt.plot(hist_C["train_loss"], color=colors["Deep+Reg"], linestyle="-", label="Deep+Reg train")
    plt.plot(hist_C["val_loss"],   color=colors["Deep+Reg"], linestyle="--", marker="^",
             markersize=4, label="Deep+Reg val")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss vs Epochs (Train & Validation)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "fig_loss_curves.png"), dpi=200)
    plt.show()


save_curves(hist_A, hist_B, hist_C)


# ============================================================
# Select best model by peak val accuracy
# ============================================================
histories = {"Shallow": hist_A, "Deep": hist_B, "Deep+Reg": hist_C}
models = {"Shallow": model_A, "Deep": model_B, "Deep+Reg": model_C}

best_name = max(histories.keys(), key=lambda k: max(histories[k]["val_acc"]))
best_model = models[best_name].to(device)
print("\nBest model:", best_name)


# ============================================================
# Predictions + metrics (Top-1, Top-3)
# ============================================================
@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        y_true.extend(y.numpy())
        y_pred.extend(preds)
        y_prob.append(probs)
    y_prob = np.vstack(y_prob)
    return np.array(y_true), np.array(y_pred), y_prob


def topk_accuracy(y_true, y_prob, k=3):
    topk = np.argsort(-y_prob, axis=1)[:, :k]
    hit = [y_true[i] in topk[i] for i in range(len(y_true))]
    return float(np.mean(hit))


y_true, y_pred, y_prob = predict_all(best_model, test_loader)
print("\nTest Accuracy (Top-1):", accuracy_score(y_true, y_pred))
print("Test Accuracy (Top-3):", topk_accuracy(y_true, y_prob, k=3))

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=train_data.classes, digits=3))


# ============================================================
# Confusion matrices for all three setups
# ============================================================
def plot_confusion_for_model(model, name):
    model = model.to(device)
    y_true_m, y_pred_m, _ = predict_all(model, test_loader)

    cm = confusion_matrix(y_true_m, y_pred_m, labels=list(range(NUM_CLASSES)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_data.classes)

    plt.figure(figsize=(9, 9))
    disp.plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"Confusion Matrix (Test) - {name}")
    plt.tight_layout()
    fname = f"fig_confusion_matrix_{name.replace('+', 'plus').replace(' ', '_')}.png"
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=200)
    plt.show()


plot_confusion_for_model(model_A, "Shallow")
plot_confusion_for_model(model_B, "Deep")
plot_confusion_for_model(model_C, "Deep+Reg")


# ============================================================
# Example predictions (best model)
# ============================================================
def denormalize(img_tensor, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]):
    img = img_tensor.detach().cpu().clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return torch.clamp(img, 0, 1)


test_plain = datasets.ImageFolder(os.path.join(OUT_DIR, "test"), transform=eval_transform)

sample_info = []
best_model.eval()
with torch.no_grad():
    for i in range(len(test_plain)):
        x, y = test_plain[i]
        logits = best_model(x.unsqueeze(0).to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred = int(probs.argmax())
        conf = float(probs[pred])
        sample_info.append((i, y, pred, conf, probs))

correct = [s for s in sample_info if s[1] == s[2]]
wrong = [s for s in sample_info if s[1] != s[2]]

correct_high = max(correct, key=lambda t: t[3])
correct_low = min(correct, key=lambda t: t[3])
wrong_high = max(wrong, key=lambda t: t[3]) if wrong else None

to_show = [correct_high, correct_low] + ([wrong_high] if wrong_high else [])

plt.figure(figsize=(12, 4))
for j, item in enumerate(to_show, start=1):
    idx, y, pred, conf, probs = item
    img, _ = test_plain[idx]
    img_vis = TF.to_pil_image(denormalize(img))

    plt.subplot(1, len(to_show), j)
    plt.imshow(img_vis)
    plt.axis("off")
    plt.title(
        f"True: {train_data.classes[y]}\n"
        f"Pred: {train_data.classes[pred]}\n"
        f"Conf: {conf:.2f}"
    )

plt.suptitle("Example Predictions (Correct High/Low Confidence + Wrong)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_example_predictions.png"), dpi=200)
plt.show()


# ============================================================
# Grad-CAM on best model
# ============================================================
def find_last_conv(model):
    last = None
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    if last is None:
        raise RuntimeError("No Conv2d found for Grad-CAM.")
    return last


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.act = None
        self.grad = None

        def fwd_hook(m, i, o):
            self.act = o

        def bwd_hook(m, grad_in, grad_out):
            self.grad = grad_out[0]

        self.h1 = target_layer.register_forward_hook(fwd_hook)
        self.h2 = target_layer.register_full_backward_hook(bwd_hook)

    def close(self):
        self.h1.remove()
        self.h2.remove()

    def generate(self, x, class_idx):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        score = logits[0, class_idx]
        score.backward()

        weights = self.grad.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.act).sum(dim=1)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.detach()


target_layer = find_last_conv(best_model)
gc = GradCAM(best_model, target_layer)


def gradcam_plot(sample_tuple, fname):
    idx, y, pred, conf, probs = sample_tuple
    img, _ = test_plain[idx]
    img_path, cls_idx = test_plain.samples[idx]
    print(f"Grad-CAM image path: {img_path}")

    x = img.unsqueeze(0).to(device)
    cam = gc.generate(x, pred)[0].cpu().numpy()
    img_vis = TF.to_pil_image(denormalize(img))

    plt.figure(figsize=(6, 6))
    plt.imshow(img_vis)
    plt.imshow(cam, cmap="jet", alpha=0.4)
    plt.axis("off")
    plt.title(
        f"Grad-CAM\nTrue: {train_data.classes[y]} | "
        f"Pred: {train_data.classes[pred]} | Conf: {conf:.2f}"
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=200)
    plt.show()


gradcam_plot(correct_high, "fig_gradcam_correct_highconf.png")
if wrong_high:
    gradcam_plot(wrong_high, "fig_gradcam_wrong_highconf.png")

gc.close()

print("\nAll key figures saved to:", FIG_DIR)
print("DONE.")
